# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset @id: {dataset.metadata.id}")
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

# Print citation for reference
if hasattr(dataset.metadata, 'citeAs'):
    print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets and their field @ids

record_sets = dataset.record_sets

print('Available Record Sets:')
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("  Field @ids:")
        for field in rs.fields:
            print(f"    - {field.id}")
    print('')
# As an example, display the first 2 records from each record set
for rs in record_sets:
    print(f"\nSample records from Record Set '{rs.name}' (@id={rs.id}):")
    try:
        for i, record in enumerate(dataset.records(record_set=rs.id)):
            if i>=2:
                break
            print(json.dumps(record, indent=2))
    except Exception as e:
        print(f"  Could not retrieve records for this record set: {e}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

The following code loads data from all available record sets.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from Record Set: '{record_set_id}' with columns: {df.columns.values}")
    else:
        print(f"No records found for Record Set: '{record_set_id}'")

# For demonstration, pick the first record set with data, show its columns and head
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in first DataFrame ({first_rs_id}): {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes.

We demonstrate this on the first record set with numeric fields.

In [ ]:
# Find a numeric field in the first DataFrame

import numpy as np

if dataframes:
    target_df = None
    rs_id_for_eda = None

    # Find the first DataFrame with at least one numeric column
    for rs_id, df in dataframes.items():
        # Use pandas' infer objects: try to coerce all columns to numeric when possible
        numeric_candidates = []
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                numeric_candidates.append(col)
        if len(numeric_candidates) > 0:
            target_df = df.copy()
            rs_id_for_eda = rs_id
            numeric_field_id = numeric_candidates[0]  # Use first numeric field
            break

    if target_df is not None:
        print(f"Using record set @id: {rs_id_for_eda}")
        print(f"First inferred numeric field: {numeric_field_id}\n")
        # Attempt filtering: take rows where numeric field is above its median
        coerced_col = pd.to_numeric(target_df[numeric_field_id], errors='coerce')
        threshold = coerced_col.median()
        filtered_df = target_df[coerced_col > threshold].copy()

        print(f"Filtered records where '{numeric_field_id}' > {threshold}")
        print(filtered_df[[numeric_field_id]].head())

        # Normalization example
        filtered_df[f"{numeric_field_id}_normalized"] = (coerced_col[filtered_df.index] - coerced_col.mean())/coerced_col.std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping example: group by a categorical field if available
        possible_group_fields = []
        for col in target_df.columns:
            if col != numeric_field_id and target_df[col].nunique() < len(target_df)//2:
                possible_group_fields.append(col)
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No categorical field suitable for grouping found.")
    else:
        print("No suitable DataFrame with numeric fields for EDA.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example histogram and scatter plot (if column available) from the numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'target_df' in locals() and target_df is not None:
    # Histogram of the numeric field
    fig, ax = plt.subplots(figsize=(6,4))
    sns.histplot(pd.to_numeric(target_df[numeric_field_id], errors='coerce').dropna(), kde=True, bins=20, ax=ax)
    ax.set_title(f"Distribution of '{numeric_field_id}'")
    ax.set_xlabel(numeric_field_id)
    plt.show()

    # If a second numeric field exists, plot scatter
    numeric_fields = [col for col in target_df.columns if col != numeric_field_id and pd.to_numeric(target_df[col], errors='coerce').notnull().sum() > 0]
    if numeric_fields:
        scatter_field = numeric_fields[0]
        fig, ax = plt.subplots(figsize=(6,4))
        sns.scatterplot(x=pd.to_numeric(target_df[numeric_field_id], errors='coerce'),
                        y=pd.to_numeric(target_df[scatter_field], errors='coerce'),
                        alpha=0.7, ax=ax)
        ax.set_xlabel(numeric_field_id)
        ax.set_ylabel(scatter_field)
        ax.set_title(f"Scatter plot: {numeric_field_id} vs. {scatter_field}")
        plt.show()
    else:
        print("No second numeric field available for scatter plot.")
else:
    print("Not enough data for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We loaded metadata, listed record sets and their field ids, loaded tabular data, performed some elementary transformations and filtering, and visualized selected numeric fields. This approach is readily extendable to deeper analysis and other datasets supporting Croissant schemas.